In [0]:
%python
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

# Source path
EVENTS_PATH = "/Volumes/shopstream/core/events/orders_stream"

# Checkpoint and schema storage
SCHEMA_PATH = "/Volumes/shopstream/core/events/autoloader_schema"
CHECKPOINT_PATH = "/Volumes/shopstream/core/events/autoloader_checkpoint"

# Define the JSON schema
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("event_ts", StringType(), True)
])

# Read streaming data using Auto Loader
orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .schema(schema)
        .load(EVENTS_PATH)
)

# Add ingestion timestamp
orders_stream = orders_stream.withColumn(
    "ingestion_ts", current_timestamp()
)

# Write stream to a Delta table
query = (
    orders_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable("shopstream.core.orders_bronze")
)

print("Auto Loader processed all available order events (re-run this cell to ingest new events).")